# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
from pathlib import Path
import pandas as pd
import os

if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/hafizahmadadilaiengineer/flyrank-ml-internship.git

repo_root = Path("flyrank-ml-internship")

df = pd.read_csv(repo_root / "data/raw/content_refresh_anonymized.csv")


In [17]:

import pandas as pd

df["update_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 180, 365, 730, 5000],
    labels=[
        "0-180",
        "181-365",
        "366-730",
        "730+"
    ]
)

signal1 = (
    df.groupby("update_bucket")
      .agg(
          n=("content_id", "count"),
          avg_trend=("trend_pct", "mean")
      )
)

display(signal1)

/tmp/ipykernel_521/1962141580.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("update_bucket")


,n,avg_trend
update_bucket,,
0-180,29826,-4.775947
181-365,169,-4.718462
366-730,5,-96.166667
730+,0,NaN


## Signal Check 1

**Signal:** Days Since Last Update → Trend

**Verdict:** MIXED

Pages that have not been updated for a long time appear to have a much lower average trend percentage. However, almost all pages (29,826) belong to the 0–180 day bucket, while only 5 pages belong to the 366–730 day bucket.

Because the older buckets contain very few observations, there is not enough evidence to confidently conclude that content age alone explains declining performance. This signal may still be useful, but it should be combined with other signals in the baseline rule.

In [18]:
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0,5,10,20,50,100],
    labels=[
        "1-5",
        "6-10",
        "11-20",
        "21-50",
        "50+"
    ]
)

signal2 = (
    df.groupby("position_bucket")
      .agg(
          n=("content_id","count"),
          avg_ctr=("ctr","mean")
      )
)

display(signal2)

/tmp/ipykernel_521/958443504.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("position_bucket")


,n,avg_ctr
position_bucket,,
1-5,3923,1.572937
6-10,9060,0.511708
11-20,7273,0.323443
21-50,7225,0.222345
50+,1299,0.152525


## Signal Check 2

**Signal:** Average Position → CTR

**Verdict:** CONFIRMED

The bucket analysis shows a clear relationship between search position and CTR. Pages ranked between positions 1–5 have the highest average CTR (1.57%), while pages ranked beyond position 50 have the lowest average CTR (0.15%).

This confirms that search position strongly influences click-through rate and supports using CTR and ranking signals when prioritizing content for review.

In [19]:
# -----------------------------
# Baseline Score
# -----------------------------

df["baseline_score"] = 0

# Stale content
df.loc[df["days_since_last_update"] >= 180, "baseline_score"] += 40

# High visibility
df.loc[df["impressions_90d"] >= 500, "baseline_score"] += 30

# Already visible in search
df.loc[
    (df["avg_position"] > 0) &
    (df["avg_position"] <= 20),
    "baseline_score"
] += 20

# Low CTR opportunity
df.loc[df["ctr"] < 0.5, "baseline_score"] += 10

In [20]:
df["reason_code"] = "general_review"

df.loc[
    (df["days_since_last_update"] >= 180) &
    (df["impressions_90d"] >= 500),
    "reason_code"
] = "stale_visible_page"

df.loc[
    (df["ctr"] < 0.5) &
    (df["avg_position"] <= 20),
    "reason_code"
] = "low_ctr_visible_page"

In [21]:
df["action"] = "Monitor"

df.loc[
    df["baseline_score"] >= 70,
    "action"
] = "Refresh Immediately"

df.loc[
    (df["baseline_score"] >= 40) &
    (df["baseline_score"] < 70),
    "action"
] = "Review Soon"

In [22]:
baseline_queue = (
    df[
        [
            "content_id",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
    .sort_values(
        "baseline_score",
        ascending=False
    )
)

baseline_queue.head(10)

,content_id,baseline_score,reason_code,action
11630,content_6226ee6adc91,100,low_ctr_visible_page,Refresh Immediately
5327,content_fe16a55cd13d,100,low_ctr_visible_page,Refresh Immediately
16751,content_cf56e2e2e282,100,low_ctr_visible_page,Refresh Immediately
22872,content_e3ff1b093148,100,low_ctr_visible_page,Refresh Immediately
12045,content_c2d929d83eaa,100,low_ctr_visible_page,Refresh Immediately
7452,content_72496874f806,100,low_ctr_visible_page,Refresh Immediately
21268,content_0a91db491d14,100,low_ctr_visible_page,Refresh Immediately
20837,content_928af3e22c80,100,low_ctr_visible_page,Refresh Immediately
26799,content_77d4d5930e5e,100,low_ctr_visible_page,Refresh Immediately
26840,content_7f116ae1f6f5,100,low_ctr_visible_page,Refresh Immediately


In [23]:
from pathlib import Path

output_dir = Path("flyrank-ml-internship/work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

baseline_queue.to_csv(
    output_dir / "baseline_action_score.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


## Baseline Rule

A transparent baseline rule was created to prioritize pages for review.

The score combines four observable signals:

- Content has not been updated for at least 180 days.
- The page has meaningful search visibility (500 or more impressions).
- The page already ranks within the top 20 search positions.
- The page has a low click-through rate (CTR below 0.5%).

Each page receives a baseline score, a reason code explaining the primary trigger, and an action label indicating whether the page should be monitored, reviewed soon, or refreshed immediately.

The ranked queue is written to `work/outputs/baseline_action_score.csv`.

In [24]:
top10 = baseline_queue.head(10)

display(top10)

,content_id,baseline_score,reason_code,action
11630,content_6226ee6adc91,100,low_ctr_visible_page,Refresh Immediately
5327,content_fe16a55cd13d,100,low_ctr_visible_page,Refresh Immediately
16751,content_cf56e2e2e282,100,low_ctr_visible_page,Refresh Immediately
22872,content_e3ff1b093148,100,low_ctr_visible_page,Refresh Immediately
12045,content_c2d929d83eaa,100,low_ctr_visible_page,Refresh Immediately
7452,content_72496874f806,100,low_ctr_visible_page,Refresh Immediately
21268,content_0a91db491d14,100,low_ctr_visible_page,Refresh Immediately
20837,content_928af3e22c80,100,low_ctr_visible_page,Refresh Immediately
26799,content_77d4d5930e5e,100,low_ctr_visible_page,Refresh Immediately
26840,content_7f116ae1f6f5,100,low_ctr_visible_page,Refresh Immediately


## 3. Review of the Top 10 Ranked Pages

The baseline ranked the following pages as the highest-priority candidates for content refresh.

All ten pages received the maximum baseline score (100) because they satisfied all conditions in the baseline rule:

- High search visibility
- Low CTR
- Good search position
- Older content

The recommendation is to review these pages first because improving already-visible pages may generate larger business impact than working on pages with little search demand.

However, every recommendation should still be reviewed by a human before taking action.

| Rank | Action              | Why Selected                                                                                           | What Would Make It Wrong                                                                                                                |
| ---- | ------------------- | ------------------------------------------------------------------------------------------------------ | --------------------------------------------------------------------------------------------------------------------------------------- |
| 1    | Refresh Immediately | High baseline score (100). The page has high visibility, a low CTR, and meets all baseline conditions. | The page may have been updated after the dataset snapshot, or the low CTR may be caused by user search intent rather than poor content. |
| 2    | Refresh Immediately | High baseline score (100). Strong visibility with a clear CTR improvement opportunity.                 | Seasonal traffic or SERP changes may explain the low CTR instead of content quality.                                                    |
| 3    | Refresh Immediately | Meets all baseline rule conditions and is prioritized for review.                                      | Recent improvements not reflected in the current dataset.                                                                               |
| 4    | Refresh Immediately | High search visibility but low CTR makes it a strong review candidate.                                 | The page may already perform well for its search intent.                                                                                |
| 5    | Refresh Immediately | Ranked highly because it satisfies every baseline rule condition.                                      | External factors such as Google's search layout may reduce CTR.                                                                         |
| 6    | Refresh Immediately | High-priority page based on the transparent baseline score.                                            | Business priorities or manual review may suggest another page is more urgent.                                                           |
| 7    | Refresh Immediately | Opportunity to improve clicks while maintaining good search visibility.                                | Low CTR could be normal for this type of content or query mix.                                                                          |
| 8    | Refresh Immediately | The rule identifies this page as an important refresh opportunity.                                     | Dataset limitations or missing recent updates could change the recommendation.                                                          |
| 9    | Refresh Immediately | High score and low CTR indicate a potential optimization opportunity.                                  | Performance may naturally recover without requiring a content refresh.                                                                  |
| 10   | Refresh Immediately | Meets all baseline criteria and is recommended for editorial review.                                   | A human reviewer may determine that no action is required after examining the page.                                                     |


In [25]:
weak_picks = (
    baseline_queue
    .sort_values("baseline_score")
    .head(10)
)

display(weak_picks)

,content_id,baseline_score,reason_code,action
11156,content_f457fc90f75b,0,general_review,Monitor
29352,content_661ea7b86418,0,general_review,Monitor
9283,content_befdfd5eb0c2,0,general_review,Monitor
21578,content_89685a4bc632,0,general_review,Monitor
21711,content_1078cff2bf78,0,general_review,Monitor
24541,content_d0548e87b05b,0,general_review,Monitor
4104,content_2e4d11f6703c,0,general_review,Monitor
2736,content_a4e083d8c665,0,general_review,Monitor
21793,content_d332d5562b63,0,general_review,Monitor
21883,content_17e6df28a5d5,0,general_review,Monitor


## 4. Weak Picks and Baseline Limitations

The baseline rule is intentionally simple and transparent, so it will not identify every worthwhile page.

Some pages receive low baseline scores because they do not meet one or more threshold conditions, even though they may still deserve review.

Examples include:

- A page with moderate impressions but a rapidly declining trend.
- A recently updated page with unusually poor engagement.
- A page with low search volume today but high future business value.
- Pages affected by seasonality or changes in search intent.
- Pages whose performance changed after the dataset snapshot.

These examples demonstrate why the baseline should be treated as a decision-support tool rather than a final decision maker. A machine learning model can combine multiple signals and discover patterns that a fixed hand-written rule cannot capture.

# 5. Self-check

- ✅ I checked two signals before building the baseline rule.
- ✅ At least one signal was linked to a real FlyRank production flag.
- ✅ I provided a verdict (CONFIRMED or MIXED) for each signal.
- ✅ I built a transparent baseline scoring rule.
- ✅ I assigned a baseline score to every content page.
- ✅ I created one reason code for every recommendation.
- ✅ I created one action label for every recommendation.
- ✅ I generated a ranked queue of pages.
- ✅ I saved the ranked queue as `work/outputs/baseline_action_score.csv`.
- ✅ I reviewed the top 10 recommendations.
- ✅ I explained what could make each recommendation wrong.
- ✅ I discussed the limitations of my baseline rule.
- ✅ I did not use any future-window or label-derived information in the scoring rule.